# Optimize SCALP-lite Parameters

Download datasets first with notebook 00. This notebook defines parameters, runs PCA then GPLVM latent Bayesian optimization, and saves the best settings for downstream notebooks.


In [1]:
from scalp_lite.notebook_utils import (
    dataset_config,
    ensure_bo_dependencies,
    make_compact_search_space,
    make_estimator,
    make_scalp_optimization_objective,
    optimization_search_space,
    run_latent_bayesopt,
    save_best_optimization_result,
)


ensure_bo_dependencies(n_threads=1)


In [2]:
selected_dataset = "pancreas"
random_state = 0
embedding_epochs = 60
invalid_score = -1e9
run_gplvm_refinement = False

pca_bo_settings = {
    "n_initial": 8,
    "latent_dim": 3,
    "n_iterations": 8,
    "embedding_model": "pca",
    "acquisition": "ei",
    "batch_size": 1,
}

gplvm_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 6,
    "embedding_model": "gplvm",
    "acquisition": "ei",
    "batch_size": 1,
}

dataset = dataset_config(selected_dataset)
raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
raw_adata = raw_estimator.to_data(dataset["input_path"])
raw_adata


AnnData object with n_obs × n_vars = 15681 × 1000
    obs: 'batch', 'study', 'cell_type', 'size_factors'

In [3]:
fixed_preprocess_params = {"max_cells": 1000}

base_preprocess_params = {
    "n_top_genes": 1500,
}

base_estimator_params = {
    "n_components": 60,
}

base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}


In [4]:
preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 2000]},
}

estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 100]},
}

graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 35]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 6]},
    "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 20]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

search_space = optimization_search_space(
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
)
search_space


{'n_top_genes': {'type': 'int', 'bounds': [500, 2000]},
 'n_components': {'type': 'int', 'bounds': [20, 100]},
 'n_neighbors': {'type': 'int', 'bounds': [5, 35]},
 'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
 'n_inter_edges': {'type': 'int', 'bounds': [1, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.1, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [3, 20]},
 'rank_correction': {'type': 'categorical', 'values': [False, True]},
 'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}}

In [5]:
objective = make_scalp_optimization_objective(
    dataset=dataset,
    raw_adata=raw_adata,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
    embedding_epochs=embedding_epochs,
    invalid_score=invalid_score,
)


In [6]:
pca_result = run_latent_bayesopt(
    objective,
    search_space,
    **pca_bo_settings,
    random_state=random_state,
    invalid_score=invalid_score,
)

pca_result["best_params"], pca_result["best_score"]


({'n_top_genes': 1255,
  'n_components': 69,
  'n_neighbors': 35,
  'intra_fraction': 0.580537494025796,
  'n_inter_edges': 5,
  'assignment_quantile': 0.9415651814089914,
  'hubness_k': 7,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': False},
 0.92915)

In [7]:
pca_result["history"].sort_values("score", ascending=False).head(10)


,iteration,phase,score,n_top_genes,n_components,n_neighbors,intra_fraction,n_inter_edges,assignment_quantile,hubness_k,rank_correction,edge_weighting,mutual_neighbors
1,0,initial,0.92915,1255,69,35,0.580537,5,0.941565,7,True,distance,False
4,0,initial,0.91330,886,69,28,0.898047,3,0.982752,9,True,distance,True
5,0,initial,0.88285,1761,75,26,0.294568,3,0.749340,18,True,binary,False
7,0,initial,0.87010,1579,68,20,0.474133,3,0.901247,7,False,distance,True
15,8,bo,0.82475,1863,87,21,0.668368,3,0.870608,15,True,binary,False
13,6,bo,0.80990,1825,86,22,0.679714,3,0.866096,15,True,binary,False
14,7,bo,0.80990,1844,86,22,0.673378,3,0.866247,15,True,binary,False
9,2,bo,0.80705,1565,79,25,0.661126,4,0.676118,14,True,binary,False
10,3,bo,0.80555,1709,83,23,0.678519,3,0.773688,15,True,binary,False
8,1,bo,0.80530,1610,79,25,0.676967,4,0.753153,15,True,binary,False


In [8]:
compact_radii = {
    "n_top_genes": 300,
    "n_components": 20,
    "n_neighbors": 6,
    "intra_fraction": 0.1,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 4,
}

compact_search_space = make_compact_search_space(
    search_space,
    pca_result["best_params"],
    compact_radii,
    fix_categoricals=True,
)
compact_search_space


{'n_top_genes': {'type': 'int', 'bounds': [955, 1555]},
 'n_components': {'type': 'int', 'bounds': [49, 89]},
 'n_neighbors': {'type': 'int', 'bounds': [29, 35]},
 'intra_fraction': {'type': 'float',
  'bounds': [0.48053749402579604, 0.680537494025796]},
 'n_inter_edges': {'type': 'int', 'bounds': [3, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.7915651814089913, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [3, 11]},
 'rank_correction': {'type': 'categorical', 'values': [True]},
 'edge_weighting': {'type': 'categorical', 'values': ['distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [False]}}

In [9]:
if run_gplvm_refinement:
    gplvm_result = run_latent_bayesopt(
        objective,
        compact_search_space,
        **gplvm_bo_settings,
        random_state=random_state + 1,
        invalid_score=invalid_score,
    )
    gplvm_summary = (gplvm_result["best_params"], gplvm_result["best_score"])
else:
    gplvm_result = None
    gplvm_summary = "skipped"

gplvm_summary


'skipped'

In [10]:
if gplvm_result is not None:
    gplvm_result["history"].sort_values("score", ascending=False).head(10)


In [11]:
optimization_results = {"pca": pca_result}
if gplvm_result is not None:
    optimization_results["gplvm"] = gplvm_result

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = save_best_optimization_result(
    dataset_name=selected_dataset,
    optimization_results=optimization_results,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
)

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params


(PosixPath('/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/scalp-lite/data/optimized_params/pancreas_graph_params.json'),
 {'n_top_genes': 1255, 'max_cells': 1000},
 {'n_components': 69},
 {'n_neighbors': 35,
  'intra_fraction': 0.580537494025796,
  'n_inter_edges': 5,
  'metric': 'euclidean',
  'assignment_quantile': 0.9415651814089914,
  'hubness_correction': 'csls',
  'hubness_k': 7,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': False,
  'neighbor_mode': 'distance',
  'symmetrize': True})